In [1]:
import boto3
import sagemaker
from pyathena import connect
import pandas as pd
from sagemaker.session import Session

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

sm = boto3.Session().client(service_name="sagemaker", region_name=region)

In [3]:
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-maintenance"

print(default_s3_bucket_name)

sagemaker-us-east-1-317166395095


In [4]:
s3_private_path_housing = "s3://{}/vehicle_maintenance.csv".format(default_s3_bucket_name)
print(s3_private_path_housing)

s3://sagemaker-us-east-1-317166395095/vehicle_maintenance.csv


In [ ]:
%store s3_private_path_housing

In [ ]:
!aws s3 cp "vehicle_maintenance_data.csv" $s3_private_path_housing

In [110]:
from sagemaker import get_execution_role

role = get_execution_role()
print(role)

arn:aws:iam::317166395095:role/LabRole


In [111]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

s3_client = boto3.client("s3", region_name=region)

vehicle_maintenance_bucket_name = default_s3_bucket_name
vehicle_maintenance = (
    "vehicle_maintenance.csv"
)


identity_vehicle_maintenance = s3_client.get_object(
    Bucket=vehicle_maintenance_bucket_name, Key=vehicle_maintenance
)


df = pd.read_csv(io.BytesIO(identity_vehicle_maintenance["Body"].read()))

In [ ]:
df.head(2)

### EDA

In [ ]:
for i in df.columns:
    if df[i].dtype != 'object':
        plt.figure(figsize=(6, 4))
        plt.hist(df[i], bins=30, edgecolor='black') 
        plt.title(f'Distribution of {i}')
        plt.xlabel(i)
        plt.ylabel('Frequency')
        plt.grid(True)
        plt.tight_layout()
        plt.show()


#### Transforming Object To Datetime

In [ ]:
df['Last_Service_Date'] = pd.to_datetime(df['Last_Service_Date'])
df['Warranty_Expiry_Date'] = pd.to_datetime(df['Warranty_Expiry_Date'])

In [ ]:
import seaborn as sns
for i in df.columns:
    if df[i].dtype == 'object':
        plt.figure(figsize=(6, 4))
        sns.countplot(df[i]) 
        plt.title(f'Distribution of {i}')
        plt.xlabel(i)
        plt.ylabel('Frequency')
        plt.grid(True)
        plt.tight_layout()
        plt.show()

#### Issues per Vehicle Type

In [ ]:
Vehicles = ['Truck', 'Van', 'Bus', 'Motorcycle', 'SUV', 'Car']
for i in Vehicles:
    subset = df[df['Vehicle_Model'] == i] 
    plt.figure(figsize=(12,5))
    sns.lineplot(data = subset, x = 'Last_Service_Date', y = 'Reported_Issues', hue= 'Fuel_Type', errorbar=None)
    plt.title(f'Reported Issues of {i}')
    plt.tight_layout()
    plt.show()

#### Maintenance Conditions

In [ ]:
sns.countplot(data= df, x = 'Vehicle_Model', hue= 'Tire_Condition')
plt.show()
sns.countplot(data= df, x = 'Vehicle_Model', hue= 'Brake_Condition')
plt.show()
sns.countplot(data= df, x = 'Vehicle_Model', hue= 'Battery_Status')
plt.show()

#### Vehicle Age and Odometer Reading per vehicle Type

In [ ]:
df.groupby('Vehicle_Model')['Vehicle_Age'].mean().plot(kind = 'bar')
plt.xticks(rotation = 45)
plt.show()
df.groupby('Vehicle_Model')['Odometer_Reading'].mean().plot(kind = 'bar')
plt.xticks(rotation = 45)
plt.show()

#### Correlation Analysis

In [ ]:
corr_mat = df.corr(numeric_only= True)
plt.figure(figsize=(12,8))
sns.heatmap(corr_mat, cmap="Blues", annot= True)

#### Feature Engineering

In [ ]:
df[['Service_History', 'Accident_History', 'Reported_Issues']]

df['Hist_Record'] = (df['Service_History'] + df['Accident_History'] + df['Reported_Issues']) /  3

df_updated = df.copy()

df_updated.drop(columns= ['Service_History', 'Accident_History', 'Reported_Issues'], axis= 1, inplace= True)

df_updated.head()

##### Encoding

In [ ]:
object_col = []

for i in df_updated.columns:
    if df[i].dtype == 'object':
        object_col.append(i)

object_col

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

for encoded in object_col:
    df_updated[encoded] = encoder.fit_transform(df_updated[encoded])

df_updated

#### Feature Engineering

In [ ]:
df_updated['service_status'] = df_updated['Tire_Condition'] + df_updated['Brake_Condition'] + df_updated['Battery_Status'] / 3

df_updated.drop(columns=['Tire_Condition', 'Brake_Condition', 'Battery_Status'], inplace= True, axis = 1)

df_updated.head(2)

#### Feature Engineering Miles Per Year

In [ ]:
df_updated['miles_per_year'] = df_updated['Mileage'] / df['Vehicle_Age']

df_updated.drop(columns=['Mileage', 'Vehicle_Age'], inplace= True, axis= 1)

df_updated.describe()

#### Correlation #2

In [ ]:
corr_mat = df_updated.corr(numeric_only= True)
plt.figure(figsize=(12,8))
sns.heatmap(corr_mat, cmap="Blues", annot= True)

#### Feature Engineering

In [ ]:
df_updated['total_hist_service'] = df_updated['Hist_Record'] + df_updated['service_status'] / 2 

df_updated.drop(columns=['Hist_Record', 'service_status'], inplace= True, axis= 1)

corr_mat = df_updated.corr(numeric_only= True)
plt.figure(figsize=(12,8))
sns.heatmap(corr_mat, cmap="Blues", annot= True)


#### Datetime Feature Engineering

In [ ]:
df_updated['last_Service_month'] = df_updated['Last_Service_Date'].dt.month

In [ ]:
df_updated['Last_Service_Date'].dt.dayofweek.value_counts()

In [ ]:
df_updated['week_number'] = df_updated['Last_Service_Date'].dt.isocalendar().week
df_updated.head(3)

In [ ]:
df_updated.drop(columns=['Last_Service_Date', 'Warranty_Expiry_Date'], axis= 1, inplace= True)

df_updated.head(3)

#### Feature Sanitization

In [ ]:
import re

def sanitize_column_name(name):
    # Replace all invalid characters with underscores
    name = re.sub(r'[^a-zA-Z0-9_-]', '_', name)
    # Truncate to 63 characters max
    name = name[:40]
    # Remove leading/trailing hyphens or underscores
    name = re.sub(r'^[-_]+|[-_]+$', '', name)
    return name

# Forcefully sanitize all columns and overwrite DataFrame
df_updated.columns = [sanitize_column_name(col) for col in df_updated.columns]

#### Ingestion Time Tracking

In [ ]:
# Time of ingestion time tracking
from time import gmtime, strftime, sleep

housing_feature_group_name = "maintenance-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

#### Feature Setup

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup

feature_group = FeatureGroup(
    name=housing_feature_group_name, sagemaker_session=feature_store_session
)

In [ ]:
df_updated['vehicle_id'] = df_updated.index.astype(str)


In [ ]:
import time

current_time_sec = int(round(time.time()))


def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


# cast object dtype to string. The SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
cast_object_to_string(df)

# record identifier and event time feature names
record_identifier_feature_name = "vehicle_id"
event_time_feature_name = "EventTime"

# append EventTime feature
df_updated[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(df), dtype="float64"
)
# Convert all boolean columns to integers
bool_cols = df_updated.select_dtypes(include='bool').columns
df_updated[bool_cols] = df_updated[bool_cols].astype(int)
# Converting categories to strings

# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
feature_group.load_feature_definitions(data_frame=df_updated)

In [ ]:
#### Creating Feature Groups in Sagemaker Feature Store

In [ ]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    role_arn=role,
    enable_online_store=True,
)


wait_for_feature_group_creation_complete(feature_group=feature_group)


In [ ]:
feature_group.describe()

In [ ]:
sagemaker_client.list_feature_groups()

In [ ]:
feature_group.ingest(data_frame=df_updated, max_workers=3, wait=True)

In [ ]:
record_identifier_value = str('100')

featurestore_runtime.get_record(
    FeatureGroupName=housing_feature_group_name,
    RecordIdentifierValueAsString=record_identifier_value,
)

# Model Preprocessing and Deployment

In [5]:
pip install awswrangler


  Using cached awswrangler-3.12.0-py3-none-any.whl.metadata (17 kB)
Using cached awswrangler-3.12.0-py3-none-any.whl (379 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
import awswrangler as wr

df = wr.athena.read_sql_query(
    sql="""
    SELECT * 
    FROM maintenance_feature_group_27_00_00_43_1748304043
    """,
    database="sagemaker_featurestore"
)

df.head()


2025-06-06 17:56:32,218	WARNING services.py:2022 -- WARNING: The object store is using /tmp instead of /dev/shm because /dev/shm has only 411021312 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=0.87gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2025-06-06 17:56:33,493	INFO worker.py:1821 -- Started a local Ray instance.


,vehicle_model,maintenance_history,fuel_type,transmission_type,engine_size,odometer_reading,owner_type,insurance_premium,fuel_efficiency,need_maintenance,miles_per_year,total_hist_service,last_service_month,week_number,vehicle_id,eventtime,write_time,api_invocation_time,is_deleted
0,2,1,2,1,2500,87800,2,14156,16.880978,1,8175.000000,5.333333,11,46,32,1.748304e+09,2025-05-27 00:06:25.362,2025-05-27 00:01:06,False
1,0,0,2,0,2500,81636,1,6220,18.709467,1,12169.800000,6.333333,8,34,3,1.748304e+09,2025-05-27 00:06:25.614,2025-05-27 00:01:06,False
2,2,1,1,1,2500,10469,1,24608,13.189379,1,7130.555556,4.000000,10,41,41,1.748304e+09,2025-05-27 00:06:25.362,2025-05-27 00:01:06,False
3,2,0,0,1,800,88378,2,27995,11.450872,1,4132.777778,1.833333,4,15,51,1.748304e+09,2025-05-27 00:06:25.362,2025-05-27 00:01:06,False
4,0,0,1,1,800,71721,2,19525,18.836759,0,7364.777778,3.000000,8,32,16770,1.748304e+09,2025-05-27 00:06:25.362,2025-05-27 00:01:07,False


In [7]:
df.drop(columns=['is_deleted','api_invocation_time','write_time','eventtime','vehicle_id'], axis= 1, inplace = True)

In [8]:
import numpy as np
!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler


  Using cached imbalanced_learn-0.13.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sklearn_compat-0.1.3-py3-none-any.whl.metadata (18 kB)
Using cached imbalanced_learn-0.13.0-py3-none-any.whl (238 kB)
Using cached sklearn_compat-0.1.3-py3-none-any.whl (18 kB)


### Re-ordering columns

In [9]:
cols = ["need_maintenance"] + [c for c in df.columns if c != "need_maintenance"]

df = df[cols]

print(df.columns.tolist())


['need_maintenance', 'vehicle_model', 'maintenance_history', 'fuel_type', 'transmission_type', 'engine_size', 'odometer_reading', 'owner_type', 'insurance_premium', 'fuel_efficiency', 'miles_per_year', 'total_hist_service', 'last_service_month', 'week_number']


In [16]:
df["need_maintenance"] = df["need_maintenance"].astype(int)


In [17]:
non_binary_col = []

for i in df.columns:
    if df[i].nunique() != 2:
        non_binary_col.append(i)
print(non_binary_col)

binary_col = []
for i in df.columns:
    if df[i].nunique() == 2:
        binary_col.append(i)
print(binary_col)



['vehicle_model', 'maintenance_history', 'fuel_type', 'engine_size', 'odometer_reading', 'owner_type', 'insurance_premium', 'fuel_efficiency', 'miles_per_year', 'total_hist_service', 'last_service_month', 'week_number']
['need_maintenance', 'transmission_type']


In [18]:
scaler = StandardScaler()

df[non_binary_col] = scaler.fit_transform(df[non_binary_col])


In [19]:
rand_split = np.random.rand(len(df))
train_list = rand_split < 0.8
val_list = (rand_split >= 0.8) & (rand_split < 0.9)
batch_list = rand_split >= 0.9

data_train = df[train_list]
data_val = df[val_list]
data_batch = df[batch_list].drop(["need_maintenance"], axis=1)
data_batch_noID = df[batch_list]


In [20]:
train_file = "train_data.csv"
data_train.to_csv(train_file, index=False, header=False)
sess.upload_data(train_file, key_prefix="{}/train".format(prefix))

validation_file = "validation_data.csv"
data_val.to_csv(validation_file, index=False, header=False)
sess.upload_data(validation_file, key_prefix="{}/validation".format(prefix))

batch_file = "batch_data.csv"
data_batch.to_csv(batch_file, index=False, header=False)
sess.upload_data(batch_file, key_prefix="{}/batch".format(prefix))

batch_file_noID = "batch_data_noID.csv"
data_batch_noID.to_csv(batch_file_noID, index=False, header=False)
sess.upload_data(batch_file_noID, key_prefix="{}/batch".format(prefix))

's3://sagemaker-us-east-1-317166395095/sagemaker-featurestore-maintenance/batch/batch_data_noID.csv'

# Training Data Preprocessing

## Deployment

In [21]:
import boto3
from sagemaker.session import s3_input
from sagemaker import image_uris
from time import gmtime, strftime

job_name        = "logreg-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
output_location = f"s3://{bucket}/{prefix}/output/{job_name}"

image = image_uris.retrieve(
    framework="linear-learner",
    region=boto3.Session().region_name,
    version="latest"
)

sm_estimator = sagemaker.estimator.Estimator(
    image_uri=image,
    role=role,                 
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=50,
    input_mode="File",
    output_path=output_location,
    sagemaker_session=sess,
)

sm_estimator.set_hyperparameters(
    predictor_type="binary_classifier",
    loss="logistic",
    l1=0.0                            
)
train_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
validation_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validation".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
data_channels = {"train": train_data, "validation": validation_data}

sm_estimator.fit(inputs=data_channels, job_name=job_name, logs=True)


INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: logreg-2025-06-06-18-00-27


2025-06-06 18:00:28 Starting - Starting the training job...
2025-06-06 18:00:53 Starting - Preparing the instances for training...
2025-06-06 18:01:30 Downloading - Downloading the training image.........
2025-06-06 18:02:46 Training - Training image download completed. Training in progress..Docker entrypoint called with argument(s): train
Running default environment configuration script
[06/06/2025 18:02:55 INFO 139779550930752] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss': 'auto', 'margin': '1.0', 'quantile': '0.5', 'loss_insensit

# Endpoint Configuration

In [22]:
sagemaker = boto3.client("sagemaker")

model_name = job_name
print(model_name)


info = sagemaker.describe_training_job(TrainingJobName=model_name)
model_data = info["ModelArtifacts"]["S3ModelArtifacts"]

primary_container = {"Image": image, "ModelDataUrl": model_data}

# Save our model to the Sagemaker Model Registry
create_model_response = sagemaker.create_model(
    ModelName=model_name, ExecutionRoleArn=role, PrimaryContainer=primary_container
)

print(create_model_response["ModelArn"])

logreg-2025-06-06-18-00-27
arn:aws:sagemaker:us-east-1:317166395095:model/logreg-2025-06-06-18-00-27


In [23]:
# Create Endpoint Configuration


# Create an endpoint config name. Here we create one based on the date  
# so it we can search endpoints based on creation time.
endpoint_config_name = 'final-project-1-endpoint-config' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())                            
                            
instance_type = 'ml.m5.xlarge'

endpoint_config_response = sagemaker.create_endpoint_config(
    EndpointConfigName=endpoint_config_name, # You will specify this name in a CreateEndpoint request.
    # List of ProductionVariant objects, one for each model that you want to host at this endpoint.
    ProductionVariants=[
        {
            "VariantName": "version-logreg1", # The name of the production variant.
            "ModelName": model_name, 
            "InstanceType": instance_type, # Specify the compute instance type.
            "InitialInstanceCount": 1 # Number of instances to launch initially.
        }
    ]
)

print(f"Created EndpointConfig: {endpoint_config_response['EndpointConfigArn']}")

Created EndpointConfig: arn:aws:sagemaker:us-east-1:317166395095:endpoint-config/final-project-1-endpoint-config2025-06-06-18-04-16


In [24]:
# Deploy our model to real-time endpoint

endpoint_name = 'logreg-endpoint' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())                            


create_endpoint_response = sagemaker.create_endpoint(
                                            EndpointName=endpoint_name, 
                                            EndpointConfigName=endpoint_config_name) 

In [25]:
from time import sleep

# Wait for endpoint to spin up

sagemaker.describe_endpoint(EndpointName=endpoint_name)

while True:
    print("Getting Job Status")
    res = sagemaker.describe_endpoint(EndpointName=endpoint_name)
    state = res["EndpointStatus"]
    
    if state == "InService":
        print("Endpoint in Service")
        break
    elif state == "Creating":
        print("Endpoint still creating...")
        sleep(60)
    else:
        print("Endpoint Creation Error - Check Sagemaker Console")
        break

Getting Job Status
Endpoint still creating...
Getting Job Status
Endpoint still creating...
Getting Job Status
Endpoint still creating...
Getting Job Status
Endpoint still creating...
Getting Job Status
Endpoint in Service


In [ ]:
# Invoke Endpoint

sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=region)

response = sagemaker_runtime.invoke_endpoint(
                            EndpointName=endpoint_name,
                            ContentType='text/csv',
                            Body=data_batch_noID.to_csv(header=None, index=False).strip('\n').split('\n')[1]
                            )
print(response['Body'].read().decode('utf-8'))

In [ ]:
sagemaker.delete_endpoint(EndpointName=endpoint_name)


# Bias Monitoring

In [ ]:
import boto3
import sagemaker
from datetime import datetime
from sagemaker import image_uris

session = sagemaker.Session()
bucket = session.default_bucket()

print("Using S3 bucket:", bucket)
prefix = "sagemaker/VehicleMaint-Monitor-" + datetime.now().strftime("%Y%m%d-%H%M%S")
print("Project prefix:", prefix)


In [ ]:
local_train   = "train_data.csv"
local_valid   = "validation_data.csv"
local_batch   = "batch_data.csv"     
local_batch_noID = "batch_data_noID.csv"  


s3_train_prefix       = f"{prefix}/baseline/train"
s3_validation_prefix  = f"{prefix}/baseline/validation"
s3_ground_truth_prefix = f"{prefix}/ground_truth"  
s3_reports_prefix     = f"{prefix}/reports"

# 3) Upload each CSV
print("Uploading train_data.csv to S3…")
session.upload_data(local_train,      bucket=bucket, key_prefix=s3_train_prefix)

print("Uploading validation_data.csv to S3…")
session.upload_data(local_valid,      bucket=bucket, key_prefix=s3_validation_prefix)

print("Uploading batch_data.csv (with labels) to S3…")
session.upload_data(local_batch,      bucket=bucket, key_prefix=s3_ground_truth_prefix)

print("Uploading batch_data_noID.csv (features only, and target) to S3…")
session.upload_data(local_batch_noID, bucket=bucket, key_prefix=f"{prefix}/batch_noID/")


In [ ]:
data_capture_prefix      = f"{prefix}/datacapture"
s3_capture_upload_path   = f"s3://{bucket}/{data_capture_prefix}"


ground_truth_upload_path = f"s3://{bucket}/{s3_validation_prefix}"
reports_prefix           = f"{prefix}/reports"
s3_report_path           = f"s3://{bucket}/{reports_prefix}"

print("Data capture path:    ", s3_capture_upload_path)
print("Ground truth path:    ", ground_truth_upload_path)
print("Monitor report path:  ", s3_report_path)

In [ ]:
header = ['need_maintenance', 'vehicle_model', 
           'maintenance_history', 'fuel_type', 'transmission_type', 
           'engine_size', 'odometer_reading', 'owner_type', 'insurance_premium',
           'fuel_efficiency', 'miles_per_year', 'total_hist_service', 'last_service_month', 'week_number']

In [ ]:
from sagemaker import get_execution_role, image_uris, Session
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig,
    SHAPConfig,
)
from sagemaker.model import Model
from sagemaker.model_monitor import (
    BiasAnalysisConfig,
    CronExpressionGenerator,
    DataCaptureConfig,
    EndpointInput,
    ExplainabilityAnalysisConfig,
    ModelBiasMonitor,
    ModelExplainabilityMonitor,
)
from sagemaker.s3 import S3Downloader, S3Uploader


# Model Bias Monitor

model_bias_monitor = ModelBiasMonitor(
    role=role,
    sagemaker_session=session,
    max_runtime_in_seconds=1800,
)



In [ ]:
# Model Baseline
baseline_job_name        = "bias-baseline-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime()) 
baseline_results_uri     = f"s3://{bucket}/{prefix}/reports/bias_baseline/{baseline_job_name}/output"
model_bias_baselining_job_result_uri = f"{baseline_results_uri}/model_bias"

model_bias_data_config = DataConfig(
    s3_data_input_path=ground_truth_upload_path,
    s3_output_path=model_bias_baselining_job_result_uri,
    label='need_maintenance',
    headers=header,
    dataset_type='text/csv',
)

In [ ]:
model_bias_data_config.__dict__

In [ ]:
# Model Bias Configuration

threshold = df['insurance_premium'].mean()

bias_config = BiasConfig(
    label_values_or_threshold=[1],            
    facet_name=[header[8]],         
    facet_values_or_threshold=[[threshold]],        

)


In [ ]:
bias_config.__dict__


In [ ]:
# Probability Threshold

model_predicted_label_config = ModelPredictedLabelConfig(
    probability_threshold=0.5,
)

In [ ]:
# Model Configuration

model_config = ModelConfig(
    model_name=model_name,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    content_type='text/csv',
    accept_type='text/csv',
)

In [ ]:
# Model Bias Monitor

model_bias_monitor.suggest_baseline(
    model_config=model_config,
    data_config=model_bias_data_config,
    bias_config=bias_config,
    model_predicted_label_config=model_predicted_label_config,
)
print(f"ModelBiasMonitor baselining job: {model_bias_monitor.latest_baselining_job_name}")

In [ ]:
# Baselining Job

model_bias_monitor.latest_baselining_job.wait(logs=False)
model_bias_constraints = model_bias_monitor.suggested_constraints()
print()
print(f"ModelBiasMonitor suggested constraints: {model_bias_constraints.file_s3_uri}")
print(S3Downloader.read_file(model_bias_constraints.file_s3_uri))

In [ ]:
from sagemaker.session import Session

session = Session()
for desc in session.sagemaker_client.list_endpoints()["Endpoints"]:
    print(desc["EndpointName"], "→", desc["EndpointStatus"])


In [ ]:
model_bias_analysis_config = None

if not model_bias_monitor.latest_baselining_job:
    model_bias_analysis_config = BiasAnalysisConfig(
        bias_config,
        headers=header,
        label='need_maintenance',
    )

In [ ]:
schedule_expression = "cron(0 */6 * * ? *)"

model_bias_monitor.create_monitoring_schedule(
    analysis_config=model_bias_analysis_config,
    output_s3_uri=s3_report_path,
    endpoint_input=EndpointInput(
        endpoint_name= endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        start_time_offset="-PT1H",
        end_time_offset="-PT0H",
        probability_threshold_attribute=0.5,
    ),
    ground_truth_input=ground_truth_upload_path,
    schedule_cron_expression=schedule_expression,
)
print(f"Model bias monitoring schedule: {model_bias_monitor.monitoring_schedule_name}")

# Model Quality Monitoring

In [26]:
%%time

from datetime import datetime, timedelta, timezone
import json
import os
import re
import boto3
from time import sleep
from threading import Thread

import pandas as pd

from sagemaker import get_execution_role, session, Session, image_uris
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.processing import ProcessingJob
from sagemaker.serializers import CSVSerializer

from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig

CPU times: user 36 μs, sys: 5 μs, total: 41 μs
Wall time: 46.3 μs


In [27]:
print("Demo Bucket:", bucket)
prefix = "sagemaker/Quality-Monitor-maintenance"

##S3 prefixes
data_capture_prefix = f"{prefix}/datacapture"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

ground_truth_upload_path = (
    f"s3://{bucket}/{prefix}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
)

reports_prefix = f"{prefix}/reports"
s3_report_path = f"s3://{bucket}/{reports_prefix}"

##Get the model monitor image
monitor_image_uri = image_uris.retrieve(framework="model-monitor", region=region)

print("Image URI:", monitor_image_uri)
print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Demo Bucket: sagemaker-us-east-1-317166395095
Image URI: 156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer
Capture path: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/datacapture
Ground truth path: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/ground_truth_data/2025-06-06-18-08-18
Report path: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/reports


In [28]:

sm = boto3.client("sagemaker", region_name= region)  # replace with your region

endpoint_name = endpoint_name  # ← your deployed endpoint’s name

# 2) Describe the endpoint to get the EndpointConfigName
resp_ep = sm.describe_endpoint(EndpointName=endpoint_name)
endpoint_config_name = resp_ep["EndpointConfigName"]

# 3) Describe the endpoint configuration to get the ModelName
resp_cfg = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
# Usually there’s a list under "ProductionVariants", and each variant has a "ModelName".
model_name = resp_cfg["ProductionVariants"][0]["ModelName"]

print("→ EndpointConfigName:", endpoint_config_name)
print("→ ModelName:", model_name)


→ EndpointConfigName: final-project-1-endpoint-config2025-06-06-18-04-16
→ ModelName: logreg-2025-06-06-18-00-27


In [29]:
S3Uploader.upload("batch_data_noID.csv", f"s3://{bucket}/test_upload")
print("Success! You are all set to proceed.")

Success! You are all set to proceed.


In [30]:
model_s3_uri = (
    "s3://sagemaker-us-east-1-317166395095/"
    "sagemaker-featurestore-maintenance/output/"
    "logreg-2025-06-06-13-10-50/"
    "logreg-2025-06-06-13-10-50/output/model.tar.gz"
)

# Create SageMaker Model entity

In [31]:
model_name = f"final-project-pred-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"

image_uri = image_uris.retrieve(framework="linear-learner", version="latest", region=boto3.Session().region_name)

model = Model(image_uri=image_uri, model_data=model_s3_uri, role=role, sagemaker_session=sess)

/tmp/ipykernel_179/854745395.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  model_name = f"final-project-pred-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


# Deploy the model with data capture enabled

In [32]:
%%time
endpoint_name = f"Final-Monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)

data_capture_config = DataCaptureConfig(
    enable_capture=True, sampling_percentage=100, destination_s3_uri=s3_capture_upload_path
)

model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
)

<timed exec>:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
INFO:sagemaker:Creating model with name: linear-learner-2025-06-06-18-08-18-912


EndpointName = Final-Monitor-2025-06-06-1808


INFO:sagemaker:Creating endpoint-config with name Final-Monitor-2025-06-06-1808
INFO:sagemaker:Creating endpoint with name Final-Monitor-2025-06-06-1808


-------!CPU times: user 796 ms, sys: 233 ms, total: 1.03 s
Wall time: 4min 2s


In [33]:
from sagemaker.predictor import Predictor

predictor = Predictor(
    endpoint_name=endpoint_name, sagemaker_session=sess, serializer=CSVSerializer()
)


# Baseline for model Quality Monitor

In [34]:
threshold = 0.5
validate_dataset = "validation_data_.predictions.csv"

In [35]:
import json
from time import sleep

limit = 200 
i = 0

with open(validate_dataset, "w") as baseline_file:
    baseline_file.write("probability,prediction,label\n") 

    with open("validation_data.csv", "r") as f:
        for row in f:
            (label, input_cols) = row.strip().split(",", 1)
            response_bytes = predictor.predict(input_cols)

            # Decode bytes → string
            response_str = response_bytes.decode("utf-8")

            # Parse JSON
            parsed = json.loads(response_str)

            # Extract the float score
            probability = parsed["predictions"][0]["score"]

            # Make a hard prediction
            prediction = "1" if probability > threshold else "0"

            #Write out probability, prediction, true label
            baseline_file.write(f"{probability},{prediction},{label}\n")

            i += 1
            if i > limit:
                break

            print(".", end="", flush=True)
            sleep(0.5)

print()
print("Done!")


........................................................................................................................................................................................................
Done!


In [36]:
!head validation_data_predictions.csv

probability,prediction,label
0.0014662740286439657,0,0
4.80253656860441e-05,0,1
0.8580560088157654,1,1
0.28499335050582886,0,1
0.10655728727579117,0,1
0.011705933138728142,0,1
0.15533293783664703,0,1
0.0007429191027767956,0,1
0.44321632385253906,0,1


In [37]:
baseline_prefix = prefix + "/baselining"
baseline_data_prefix = baseline_prefix + "/data"
baseline_results_prefix = baseline_prefix + "/results"

baseline_data_uri = f"s3://{bucket}/{baseline_data_prefix}"
baseline_results_uri = f"s3://{bucket}/{baseline_results_prefix}"
print(f"Baseline data uri: {baseline_data_uri}")
print(f"Baseline results uri: {baseline_results_uri}")

Baseline data uri: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/baselining/data
Baseline results uri: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/baselining/results


In [38]:
baseline_dataset_uri = S3Uploader.upload(f"{validate_dataset}", baseline_data_uri)
baseline_dataset_uri

's3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/baselining/data/validation_data_.predictions.csv'

# Create a baselining job with validation dataset predictions

In [39]:
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor import EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

In [40]:
# Create the model quality monitoring object
maintenance_model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sess,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [41]:
baseline_job_name = f"maintenance-model-baseline-job-{datetime.utcnow():%Y-%m-%d-%H%M}"


/tmp/ipykernel_179/2097779891.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  baseline_job_name = f"maintenance-model-baseline-job-{datetime.utcnow():%Y-%m-%d-%H%M}"


In [42]:
# Execute the baseline suggestion job.
# You will specify problem type, in this case Binary Classification, and provide other required attributes.
job = maintenance_model_quality_monitor.suggest_baseline(
    job_name=baseline_job_name,
    baseline_dataset=baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
)
job.wait(logs=False)

INFO:sagemaker:Creating processing-job with name maintenance-model-baseline-job-2025-06-06-1814


...........................................................!

In [43]:
baseline_job = maintenance_model_quality_monitor.latest_baselining_job


In [44]:
binary_metrics = baseline_job.baseline_statistics().body_dict["binary_classification_metrics"]
pd.json_normalize(binary_metrics).T

,0
confusion_matrix.0.0,38
confusion_matrix.0.1,2
confusion_matrix.1.0,151
confusion_matrix.1.1,10
recall.value,0.062112
recall.standard_deviation,0.006692
precision.value,0.833333
precision.standard_deviation,0.039457
accuracy.value,0.238806
accuracy.standard_deviation,0.011815


In [45]:
pd.DataFrame(baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T


,threshold,comparison_operator
recall,0.062112,LessThanThreshold
precision,0.833333,LessThanThreshold
accuracy,0.238806,LessThanThreshold
true_positive_rate,0.062112,LessThanThreshold
true_negative_rate,0.95,LessThanThreshold
false_positive_rate,0.05,GreaterThanThreshold
false_negative_rate,0.937888,GreaterThanThreshold
auc,0.621118,LessThanThreshold
f0_5,0.239234,LessThanThreshold
f1,0.115607,LessThanThreshold


# Continous Monitoring

In [46]:
def invoke_endpoint(ep_name, file_name):
    with open(file_name, "r") as f:
        i = 0
        for row in f:
            payload = row.rstrip("\n")
            response = session.sagemaker_runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Body=payload,
                InferenceId=str(i),  # unique ID per row
            )["Body"].read()
            i += 1
            sleep(1)


def invoke_endpoint_forever():
    while True:
        try:
            invoke_endpoint(endpoint_name, "test_data/test-dataset-input-cols.csv")
        except session.sagemaker_runtime_client.exceptions.ValidationError:
            pass


thread = Thread(target=invoke_endpoint_forever)
thread.start()

Exception in thread Thread-6 (invoke_endpoint_forever):
Traceback (most recent call last):
  File "/tmp/ipykernel_179/2099417762.py", line 19, in invoke_endpoint_forever
  File "/tmp/ipykernel_179/2099417762.py", line 2, in invoke_endpoint
  File "/opt/conda/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 324, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'test_data/test-dataset-input-cols.csv'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/opt/conda/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_179/

In [47]:
print("Waiting for captures to show up", end="")
for _ in range(120):
    capture_files = sorted(S3Downloader.list(f"{s3_capture_upload_path}/{endpoint_name}"))
    if capture_files:
        capture_file = S3Downloader.read_file(capture_files[-1]).split("\n")
        capture_record = json.loads(capture_file[0])
        if "inferenceId" in capture_record["eventMetadata"]:
            break
    print(".", end="", flush=True)
    sleep(1)
print()
print("Found Capture Files:")
print("\n ".join(capture_files[-3:]))



Waiting for captures to show up........................................................................................................................
Found Capture Files:
s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/datacapture/Final-Monitor-2025-06-06-1808/AllTraffic/2025/06/06/18/12-21-675-76863d5d-170a-44bb-b716-5be1a6d993cc.jsonl
 s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/datacapture/Final-Monitor-2025-06-06-1808/AllTraffic/2025/06/06/18/13-21-892-273e7b60-ce3e-4d19-9d8a-bf4e24aa8e3e.jsonl


In [48]:
print("\n".join(capture_file[-3:-1]))

{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"0.874310987264832,1.2295098552373167,-1.2217747289386287,0,0.7069118079254093,0.9828521302508908,-1.231914205169946,0.25177605396955105,1.0880299360558872,3.480636005980886,-1.3988528899256893,0.6309610548244904,0.5480415397099164","encoding":"CSV"},"endpointOutput":{"observedContentType":"application/json","mode":"OUTPUT","data":"{\"predictions\": [{\"score\": 3.768504029721953e-05, \"predicted_label\": 0}]}","encoding":"JSON"}},"eventMetadata":{"eventId":"571f5c45-48e5-417c-90ef-43be836fa282","inferenceTime":"2025-06-06T18:14:03Z"},"eventVersion":"0"}
{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"-0.878634920503995,0.003921881515908467,-1.2217747289386287,0,-0.8862796781992701,-0.027297529192432952,1.2252797814061906,-0.7312897457862261,0.12415775184701458,-0.6883333252536342,-0.25793803124951054,0.6309610548244904,0.5480415397099164","encoding":"CSV"},"

In [49]:
print(json.dumps(capture_record, indent=2))

{
  "captureData": {
    "endpointInput": {
      "observedContentType": "text/csv",
      "mode": "INPUT",
      "data": "0.874310987264832,1.2295098552373167,0.0014189046149221548,1,0.7069118079254093,-1.44032341926819,1.2252797814061906,-0.22293632842202196,-1.3899558326986812,0.282243971522045,0.12236692164254911,-0.24500064874318966,-0.324903386618444",
      "encoding": "CSV"
    },
    "endpointOutput": {
      "observedContentType": "application/json",
      "mode": "OUTPUT",
      "data": "{\"predictions\": [{\"score\": 0.06541630625724792, \"predicted_label\": 1}]}",
      "encoding": "JSON"
    }
  },
  "eventMetadata": {
    "eventId": "9a3546b2-2759-4c78-a5f1-6c026cb31c6a",
    "inferenceTime": "2025-06-06T18:13:21Z"
  },
  "eventVersion": "0"
}


# Generating Ground Truths

In [55]:


import random


def ground_truth_with_id(inference_id):
    random.seed(inference_id)  # to get consistent results
    rand = random.random()
    return {
        "groundTruthData": {
            "data": "1" if rand < 0.7 else "0",  # randomly generate positive labels 70% of the time
            "encoding": "CSV",
        },
        "eventMetadata": {
            "eventId": str(inference_id),
        },
        "eventVersion": "0",
    }


def upload_ground_truth(records, upload_time):
    fake_records = [json.dumps(r) for r in records]
    data_to_upload = "\n".join(fake_records)
    target_s3_uri = f"{ground_truth_upload_path}/{upload_time:%Y/%m/%d/%H/%M%S}.jsonl"
    print(f"Uploading {len(fake_records)} records to", target_s3_uri)
    S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)




In [76]:
NUM_GROUND_TRUTH_RECORDS = 334  


def generate_fake_ground_truth_forever():
    j = 0
    while True:
        fake_records = [ground_truth_with_id(i) for i in range(NUM_GROUND_TRUTH_RECORDS)]
        upload_ground_truth(fake_records, datetime.utcnow())
        j = (j + 1) % 5
        sleep(60 * 60)  # do this once an hour


gt_thread = Thread(target=generate_fake_ground_truth_forever)
gt_thread.start()



Uploading 334 records to s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/ground_truth_data/2025-06-06-18-08-18/2025/06/06/19/2813.jsonl


/tmp/ipykernel_179/2751130122.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  upload_ground_truth(fake_records, datetime.utcnow())


# Creating a Monitoring Scheduel

In [57]:
##Monitoring schedule name
maintenance_monitor_schedule_name = (
    f"log-reg-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"
)

/tmp/ipykernel_179/187940405.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"log-reg-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"


In [58]:
# Create an enpointInput
endpointInput = EndpointInput(
    endpoint_name=predictor.endpoint_name,
    probability_attribute="0",
    probability_threshold_attribute=0.5,
    destination="/opt/ml/processing/input_data",
)



In [ ]:
# Create the monitoring schedule to execute every hour.
from sagemaker.model_monitor import CronExpressionGenerator

response = maintenance_model_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=maintenance_monitor_schedule_name,
    endpoint_input=endpointInput,
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    ground_truth_input=ground_truth_upload_path,
    constraints=baseline_job.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)



In [61]:
maintenance_model_quality_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:317166395095:monitoring-schedule/log-reg-monitoring-schedule-2025-06-06-1839',
 'MonitoringScheduleName': 'log-reg-monitoring-schedule-2025-06-06-1839',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2025, 6, 6, 18, 42, 18, 196000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2025, 6, 6, 18, 42, 24, 505000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'cron(0 * ? * * *)'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2025-06-06-18-42-17-485',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'Final-Monitor-2025-06-06-1808',
 'ResponseMetadata': {'RequestId': '6d3e7c3f-c9f8-42a3-b25d-9ebb6ff04f7e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '6d3e7c3f-c9f8-42a3-b25d-9ebb6ff04f7e',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '595',
   'date': 'Fri

In [62]:
executions = maintenance_model_quality_monitor.list_executions()
executions

[]

In [64]:
print("Waiting for first execution", end="")
while True:
    execution = maintenance_model_quality_monitor.describe_schedule().get(
        "LastMonitoringExecutionSummary"
    )
    if execution:
        break
    print(".", end="", flush=True)
    sleep(10)
print()
print("Execution found!")

Waiting for first execution....................................................................................................................................................
Execution found!


In [65]:
while not executions:
    executions = maintenance_model_quality_monitor.list_executions()
    print(".", end="", flush=True)
    sleep(10)
latest_execution = executions[-1]
latest_execution.describe()



.....................

{'ProcessingInputs': [{'InputName': 'groundtruth_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/ground_truth_data/2025-06-06-18-08-18/2025/06/06/18',
    'LocalPath': '/opt/ml/processing/groundtruth/2025/06/06/18',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}},
  {'InputName': 'endpoint_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/datacapture/Final-Monitor-2025-06-06-1808/AllTraffic/2025/06/06/18',
    'LocalPath': '/opt/ml/processing/input_data/Final-Monitor-2025-06-06-1808/AllTraffic/2025/06/06/18',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}}],
 'ProcessingOutputConfig': {'Outputs': [{'OutputName': 'result',
    'S3Output'

In [67]:
status = execution["MonitoringExecutionStatus"]

while status in ["Pending", "InProgress"]:
    print("Waiting for execution to finish", end="")
    latest_execution.wait(logs=False)
    latest_job = latest_execution.describe()
    print()
    print(f"{latest_job['ProcessingJobName']} job status:", latest_job["ProcessingJobStatus"])
    print(
        f"{latest_job['ProcessingJobName']} job exit message, if any:",
        latest_job.get("ExitMessage"),
    )
    print(
        f"{latest_job['ProcessingJobName']} job failure reason, if any:",
        latest_job.get("FailureReason"),
    )
    sleep(
        30
    )  # model quality executions consist of two Processing jobs, wait for second job to start
    latest_execution = maintenance_model_quality_monitor.list_executions()[-1]
    execution = maintenance_model_quality_monitor.describe_schedule()["LastMonitoringExecutionSummary"]
    status = execution["MonitoringExecutionStatus"]

print("Execution status is:", status)

if status != "Completed":
    print(execution)
    print(
        "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
    )



Waiting for execution to finish...............................................!
groundtruth-merge-202506061900-79c54fef7be644926ed1fb5a job status: Completed
groundtruth-merge-202506061900-79c54fef7be644926ed1fb5a job exit message, if any: None
groundtruth-merge-202506061900-79c54fef7be644926ed1fb5a job failure reason, if any: None
Execution status is: Failed
{'MonitoringScheduleName': 'log-reg-monitoring-schedule-2025-06-06-1839', 'ScheduledTime': datetime.datetime(2025, 6, 6, 19, 0, tzinfo=tzlocal()), 'CreationTime': datetime.datetime(2025, 6, 6, 19, 8, 44, 301000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2025, 6, 6, 19, 17, 11, 433000, tzinfo=tzlocal()), 'MonitoringExecutionStatus': 'Failed', 'ProcessingJobArn': 'arn:aws:sagemaker:us-east-1:317166395095:processing-job/groundtruth-merge-202506061900-79c54fef7be644926ed1fb5a', 'EndpointName': 'Final-Monitor-2025-06-06-1808', 'FailureReason': 'Job inputs had no data'}
====STOP==== 
 No completed executions to inspect fu

In [69]:
latest_execution = maintenance_model_quality_monitor.list_executions()[-1]
report_uri = latest_execution.describe()["ProcessingOutputConfig"]["Outputs"][0]["S3Output"][
    "S3Uri"
]
print("Report Uri:", report_uri)

Report Uri: s3://sagemaker-us-east-1-317166395095/sagemaker/Quality-Monitor-maintenance/baselining/results/merge


# Cloudwatch

In [ ]:
# Create CloudWatch client
cw_client = boto3.Session().client("cloudwatch")

namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_dimensions = [
    {"Name": "Endpoint", "Value": endpoint_name},
    {"Name": "MonitoringSchedule", "Value": maintenance_scheduel_name},

In [ ]:
# List metrics through the pagination interface
paginator = cw_client.get_paginator("list_metrics")

for response in paginator.paginate(Dimensions=cw_dimensions, Namespace=namespace):
    model_quality_metrics = response["Metrics"]
    for metric in model_quality_metrics:
        print(metric["MetricName"])

